# QDIST v4.1 — blinded human-review adjudication

Run only after two reviewers have independently completed every item. This notebook validates the sheets, quantifies exact-label and binary hard-clip agreement, lists disagreements, and reports reviewer-positive fractions by sampling stratum. It does not finalize or freeze features.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'config' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not locate config/project.yaml.')

ROOT = find_project_root(Path.cwd())
for source in [ROOT / 'src', ROOT / 'src']:
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))
from paper1_qc_reviewed.qdist_v410_cohort import CohortPaths, adjudicate_blind_review
paths = CohortPaths.from_project_root(ROOT)
manifest_path = paths.output_root / 'manifests' / 'qdist_v410_candidate_cohort_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['candidate_only'] is True and manifest['freeze_allowed'] is False
print('Project root:', ROOT)

In [ ]:
review_root = paths.output_root / 'blind_review'
REVIEWER_1 = review_root / 'reviewer_1_COMPLETED.csv'
REVIEWER_2 = review_root / 'reviewer_2_COMPLETED.csv'
ADJUDICATION = review_root / 'adjudication_COMPLETED.csv'
for path in [REVIEWER_1, REVIEWER_2]:
    if not path.exists():
        raise FileNotFoundError(f'Completed reviewer sheet not found: {path}')

In [ ]:
review = adjudicate_blind_review(
    paths.output_root, REVIEWER_1, REVIEWER_2,
    adjudication=ADJUDICATION if ADJUDICATION.exists() else None,
)
display(review['summary'])
display(review['strata'])
display(review['review_checks'])
display(Markdown(f"**Exact-label disagreements requiring adjudication: {len(review['exact_disagreements'])}**"))
display(review['exact_disagreements'])
if not ADJUDICATION.exists() and len(review['exact_disagreements']):
    print('Filter adjudication_TEMPLATE.csv to these IDs, complete it, save as adjudication_COMPLETED.csv, and rerun.')

## Required scientific interpretation

Before finalization, inspect every disagreement and summarize false-positive morphology, missed or ambiguous morphology, polarity-specific behavior, low-level-path examples, valid-zero reviewer positives, and any systematic dependence on codec, sample rate, exposure, or participant clustering. Agreement alone does not validate the construct. Final feature decisions and manuscript reconciliation remain a separate governed step.

In [ ]:
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = 'PENDING'
assert PUBLISH_AND_FREEZE is False
assert SCIENTIFIC_REVIEW_DECISION == 'PENDING'
print('Adjudication evidence saved. Feature finalization and freeze remain blocked.')